# CNN Lab 2 — Emoji Emotion Detective

**Generate and classify 2,400 colorful faces**  
**Difficulty:** Beginner–Intermediate

All model inputs have shape **`(64, 64, 3)`**. The dataset contains 2,400–3,000 images and is sized for free Google Colab.

### Student TODOs
1. Normalize the images.
2. Define and compile a CNN.
3. Train the CNN.
4. Test the CNN.

Complete code is supplied for visualization, learning curves, predictions, a confusion matrix, and Grad-CAM.

## Colab use
Run cells from top to bottom. CPU works; a free GPU is optional. Downloaded datasets are cached only for the current Colab runtime. If the session disconnects, rerun earlier cells.

In [ ]:
# Free Google Colab CPU is sufficient. A free GPU is optional.
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

keras.utils.set_random_seed(42)
print('TensorFlow:', tf.__version__)
print('Target image shape: (64, 64, 3)')
print('Devices:', tf.config.list_physical_devices())


## Load or generate the dataset
The data is kept as `uint8` until the student normalization step. A stratified split preserves class proportions in training and test data.

In [ ]:
from PIL import Image, ImageDraw

def make_emoji_dataset(samples_per_class=600, seed=42):
    rng = np.random.default_rng(seed)
    class_names = ['happy', 'sad', 'surprised', 'angry']
    images, labels = [], []

    for label, expression in enumerate(class_names):
        for _ in range(samples_per_class):
            bg = tuple(int(v) for v in rng.integers(15, 225, size=3))
            face = tuple(int(v) for v in rng.integers(105, 256, size=3))
            canvas = Image.new('RGB', (64, 64), bg)
            draw = ImageDraw.Draw(canvas)
            r = int(rng.integers(21, 27))
            cx, cy = int(rng.integers(29, 36)), int(rng.integers(29, 36))
            draw.ellipse([cx-r, cy-r, cx+r, cy+r], fill=face, outline=(25,25,25), width=2)

            ey = cy - int(rng.integers(6, 11))
            eo, er = int(rng.integers(7, 11)), int(rng.integers(2, 4))
            for ex in (cx-eo, cx+eo):
                draw.ellipse([ex-er, ey-er, ex+er, ey+er], fill=(20,20,20))

            mw, mh = int(rng.integers(18, 25)), int(rng.integers(10, 16))
            if expression == 'happy':
                draw.arc([cx-mw//2, cy-3, cx+mw//2, cy+mh], 10, 170, fill=(20,20,20), width=3)
            elif expression == 'sad':
                draw.arc([cx-mw//2, cy+4, cx+mw//2, cy+mh+10], 190, 350, fill=(20,20,20), width=3)
            elif expression == 'surprised':
                mr = int(rng.integers(5, 8))
                draw.ellipse([cx-mr, cy+4, cx+mr, cy+4+2*mr], fill=(20,20,20))
            else:
                by = ey - 6
                draw.line([cx-eo-5, by-2, cx-eo+4, by+2], fill=(20,20,20), width=3)
                draw.line([cx+eo-4, by+2, cx+eo+5, by-2], fill=(20,20,20), width=3)
                draw.line([cx-mw//2, cy+13, cx+mw//2, cy+9], fill=(20,20,20), width=3)

            canvas = canvas.rotate(float(rng.uniform(-8, 8)), resample=Image.Resampling.BILINEAR, fillcolor=bg)
            arr = np.asarray(canvas, dtype=np.float32)
            arr = np.clip(arr + rng.normal(0, 5, arr.shape), 0, 255).astype(np.uint8)
            images.append(arr); labels.append(label)

    images = np.asarray(images, dtype=np.uint8)
    labels = np.asarray(labels, dtype=np.int64)
    order = rng.permutation(len(images))
    return images[order], labels[order], class_names

all_images, all_labels, class_names = make_emoji_dataset(600)
x_train_raw, x_test_raw, y_train, y_test = train_test_split(
    all_images,
    all_labels,
    test_size=0.20,
    random_state=42,
    stratify=all_labels,
)
input_shape = (64, 64, 3)
num_classes = len(class_names)


In [ ]:
print('Training images:', x_train_raw.shape)
print('Test images:', x_test_raw.shape)
print('Raw dtype:', x_train_raw.dtype)
print('Raw range:', int(x_train_raw.min()), 'to', int(x_train_raw.max()))
print('Classes:', class_names)
print('Training class counts:', np.bincount(y_train, minlength=num_classes))
print('Test class counts:', np.bincount(y_test, minlength=num_classes))


## Explore first
Random guessing gives about **25%** accuracy. Look for useful patterns, distracting backgrounds, similar classes, and possible shortcuts.

In [ ]:
def show_examples(images, labels, class_names, rows=3, cols=5):
    total = min(rows * cols, len(images))
    plt.figure(figsize=(12, 7))
    for i in range(total):
        plt.subplot(rows, cols, i + 1)
        plt.imshow(images[i])
        plt.title(class_names[int(labels[i])])
        plt.axis('off')
    plt.tight_layout()
    plt.show()

show_examples(x_train_raw, y_train, class_names)


## Mathematics

Every image has shape

$
64\times64\times3,
$

so it contains $64\cdot64\cdot3=12{,}288$ pixel-channel values.

### Normalization

$
x_{\text{normalized}}=\frac{x_{\text{raw}}}{255}.
$

The visual image does not change; only the numerical scale changes.

### Convolution

A filter computes a local weighted sum:

$
z_{i,j,k}=b_k+\sum_{u,v,c}W_{u,v,c,k}X_{i+u,j+v,c}.
$

Early filters may detect edges and colors. Deeper filters combine them into textures, shapes, and object parts.

### ReLU and pooling

$
\operatorname{ReLU}(z)=\max(0,z).
$

A $2\times2$ max-pooling operation can reduce $64\times64$ to $32\times32$, making later computation cheaper.

### Softmax and loss

$
p_k=\frac{e^{s_k}}{\sum_j e^{s_j}},
\qquad
\mathcal{L}=-\log p_y.
$

Softmax probabilities add to 1. Cross-entropy is small for a confident correct answer and large for a confident wrong answer.

### Learning

$
\theta\leftarrow\theta-\eta\nabla_\theta\mathcal{L}.
$

The gradient indicates how the loss changes, and the learning rate $\eta$ controls the update size.


## STUDENT TODO 1 — Normalize
Create `x_train` and `x_test`. Keep labels as integer class IDs.

In [ ]:
# STUDENT TODO 1: NORMALIZE THE PIXELS
# Create float32 arrays x_train and x_test with values from 0 to 1.
# Mathematical hint: normalized_pixel = raw_pixel / 255.0

x_train = None
x_test = None

print('Replace None with your normalization code.')


In [ ]:
def check_normalization(x_train, x_test):
    if x_train is None or x_test is None:
        print('Normalization is not complete.')
        return False
    ok = (
        x_train.shape[1:] == (64, 64, 3)
        and x_test.shape[1:] == (64, 64, 3)
        and x_train.dtype == np.float32
        and x_test.dtype == np.float32
        and 0 <= float(x_train.min()) <= float(x_train.max()) <= 1
        and 0 <= float(x_test.min()) <= float(x_test.max()) <= 1
    )
    print('Shape:', x_train.shape)
    print('dtype:', x_train.dtype)
    print('range:', float(x_train.min()), 'to', float(x_train.max()))
    print('Passed!' if ok else 'Check shape, dtype, and range again.')
    return ok

check_normalization(x_train, x_test)


## Optional augmentation
Augmentation creates small training-time transformations and can improve robustness. Test data must not be augmented.

In [ ]:
# Optional: students may include this object near the start of their CNN.
data_augmentation = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.08),
], name='data_augmentation')
print('Optional augmentation block is ready.')


## STUDENT TODO 2 — Design the CNN
Try two convolution blocks. Think about whether horizontal flipping is appropriate. Your architecture is a testable hypothesis, not a guaranteed answer.

In [ ]:
# STUDENT TODO 2: DEFINE AND COMPILE A CNN
# Required input shape: (64, 64, 3)
# Suggested budget: 2-3 Conv2D layers, 8-64 filters, under 1M parameters.
# Useful layers:
#   layers.Input(shape=input_shape)
#   data_augmentation
#   layers.Conv2D(16, 3, padding='same', activation='relu')
#   layers.MaxPooling2D(2)
#   layers.Dropout(0.2)
#   layers.GlobalAveragePooling2D()
#   layers.Flatten()
#   layers.Dense(32, activation='relu')
#   layers.Dense(num_classes, activation='softmax')
# Compile with Adam, sparse categorical cross-entropy, and accuracy.

model = None
print('Define and compile your model.')


In [ ]:
def check_model(model):
    if model is None:
        print('Model is not defined.')
        return False
    has_conv = any(isinstance(layer, layers.Conv2D) for layer in model.layers)
    correct_input = tuple(model.input_shape[1:]) == input_shape
    correct_output = model.output_shape[-1] == num_classes
    params = model.count_params()
    print('Has Conv2D:', has_conv)
    print('Input:', model.input_shape, 'Output:', model.output_shape)
    print('Parameters:', f'{params:,}')
    if params > 1_000_000:
        print('Warning: use fewer filters or GlobalAveragePooling2D for free Colab.')
    ok = has_conv and correct_input and correct_output
    if ok:
        model.summary()
    else:
        print('Revise the input, convolution, or final output layer.')
    return ok

check_model(model)


## STUDENT TODO 3 — Train
An epoch is one full pass through training data. Validation data measures progress without updating weights.

In [ ]:
# STUDENT TODO 3: TRAIN THE CNN
# Use model.fit(x_train, y_train, validation_split=0.20, ...)
# Suggested: 6-12 epochs and batch_size 32 or 64.
# Save the returned object as history.

history = None
print('Train the model and save the result as history.')


## Interpret the curves
Training and validation improving together is healthy. A growing gap often indicates overfitting. Validation loss rising while training loss falls is another warning sign.

In [ ]:
def plot_learning_curves(history):
    if history is None:
        print('Train the model first.')
        return
    h = history.history
    epochs = range(1, len(h['loss']) + 1)

    plt.figure(figsize=(8, 5))
    plt.plot(epochs, h['accuracy'], marker='o', label='Training accuracy')
    plt.plot(epochs, h['val_accuracy'], marker='o', label='Validation accuracy')
    plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.title('Accuracy curves')
    plt.grid(alpha=0.3); plt.legend(); plt.show()

    plt.figure(figsize=(8, 5))
    plt.plot(epochs, h['loss'], marker='o', label='Training loss')
    plt.plot(epochs, h['val_loss'], marker='o', label='Validation loss')
    plt.xlabel('Epoch'); plt.ylabel('Cross-entropy loss'); plt.title('Loss curves')
    plt.grid(alpha=0.3); plt.legend(); plt.show()

plot_learning_curves(history)


## STUDENT TODO 4 — Test
Use the test set as a final exam, not as repeated feedback for redesigning the model.

In [ ]:
# STUDENT TODO 4: TEST THE CNN
# Use model.evaluate(x_test, y_test, verbose=0).
# Save the outputs as test_loss and test_accuracy, then print accuracy as a percent.

test_loss = None
test_accuracy = None
print('Evaluate the model on the test set.')


## Individual predictions
Softmax confidence is not certainty. A confident prediction can still be wrong.

In [ ]:
def show_predictions(model, images, labels, class_names, count=12):
    if model is None or images is None:
        print('Finish normalization and training first.')
        return
    count = min(count, len(images))
    probs = model.predict(images[:count], verbose=0)
    preds = np.argmax(probs, axis=1)
    cols, rows = 4, int(np.ceil(count / 4))
    plt.figure(figsize=(12, 3.2 * rows))
    for i in range(count):
        plt.subplot(rows, cols, i + 1)
        plt.imshow(np.clip(images[i], 0, 1))
        p, t = int(preds[i]), int(labels[i])
        mark = '✓' if p == t else '✗'
        plt.title(f'{mark} Pred: {class_names[p]}\nTrue: {class_names[t]}\nConfidence: {100*probs[i,p]:.1f}%')
        plt.axis('off')
    plt.tight_layout(); plt.show()

if x_test is None:
    print('Normalize the data first.')
else:
    show_predictions(model, x_test, y_test, class_names)


## Confusion matrix\nEntry $C_{ij}$ counts images whose true class is $i$ and predicted class is $j$. Diagonal values are correct; off-diagonal values reveal specific mistakes.

In [ ]:
def plot_confusion(model, images, labels, class_names):
    if model is None or images is None:
        print('Finish normalization and training first.')
        return
    preds = np.argmax(model.predict(images, verbose=0), axis=1)
    matrix = confusion_matrix(labels, preds)
    size = max(7, 0.65 * len(class_names))
    fig, ax = plt.subplots(figsize=(size, size))
    ConfusionMatrixDisplay(matrix, display_labels=class_names).plot(
        ax=ax, cmap='Blues', xticks_rotation=45, colorbar=False
    )
    plt.title('Confusion Matrix'); plt.tight_layout(); plt.show()

if x_test is None:
    print('Normalize the data first.')
else:
    plot_confusion(model, x_test, y_test, class_names)


## Grad-CAM mathematics

Grad-CAM estimates which spatial locations supported class $c$. For feature map $A^k$,

$
\alpha_k^c=\frac{1}{Z}\sum_{i,j}\frac{\partial s^c}{\partial A_{i,j}^k},
$

and

$
L_{\text{Grad-CAM}}^c=\operatorname{ReLU}\left(\sum_k\alpha_k^c A^k\right).
$

Bright areas strongly supported the selected class. This is useful evidence, but it is not proof that the CNN understands the image like a human.


In [ ]:
def last_conv_layer(model):
    """Return the final Conv2D layer in a model."""
    for layer in reversed(model.layers):
        if isinstance(layer, layers.Conv2D):
            return layer

    raise ValueError("Grad-CAM requires at least one Conv2D layer.")


def build_gradcam_model(model, image_shape):
    """
    Create a functional model for Grad-CAM without using model.output.

    Some Keras Sequential models do not expose model.output as a symbolic
    tensor. We create a fresh symbolic input and pass it through the trained
    layers while reusing all learned weights.
    """
    target_conv_layer = last_conv_layer(model)

    symbolic_input = keras.Input(
        shape=image_shape,
        name="gradcam_input"
    )
    x = symbolic_input
    convolution_features = None

    for layer in model.layers:
        # Disable Dropout and random augmentation while explaining an image.
        try:
            x = layer(x, training=False)
        except TypeError:
            x = layer(x)

        if layer is target_conv_layer:
            convolution_features = x

    if convolution_features is None:
        raise ValueError(
            "The final Conv2D layer was not connected to the prediction."
        )

    return keras.Model(
        inputs=symbolic_input,
        outputs=[convolution_features, x],
        name="gradcam_model"
    )


def make_gradcam_heatmap(image_batch, model, class_index=None):
    """Compute a normalized Grad-CAM heatmap for one image."""
    grad_model = build_gradcam_model(
        model,
        image_shape=tuple(image_batch.shape[1:])
    )

    with tf.GradientTape() as tape:
        features, predictions = grad_model(
            image_batch,
            training=False
        )

        if class_index is None:
            class_index = tf.argmax(predictions[0])

        selected_score = predictions[:, class_index]

    gradients = tape.gradient(
        selected_score,
        features
    )

    if gradients is None:
        raise RuntimeError(
            "Gradients could not be computed. Check that the CNN's final "
            "Conv2D layer is connected to the output."
        )

    # Average gradient importance over height, width, and batch.
    weights = tf.reduce_mean(
        gradients,
        axis=(0, 1, 2)
    )

    # Weighted sum of the last convolution feature maps.
    heatmap = tf.reduce_sum(
        features[0] * weights,
        axis=-1
    )

    # Keep positive evidence and scale the heatmap to [0, 1].
    heatmap = tf.maximum(heatmap, 0)
    heatmap = heatmap / (
        tf.reduce_max(heatmap) + keras.backend.epsilon()
    )

    return heatmap.numpy(), int(class_index)


def show_gradcam(model, image, class_names, true_label=None):
    """Show the image, Grad-CAM heatmap, and an overlay."""
    if model is None:
        print("Define and train the model first.")
        return

    image_batch = np.expand_dims(
        image,
        axis=0
    ).astype("float32")

    probabilities = model.predict(
        image_batch,
        verbose=0
    )[0]

    predicted_index = int(np.argmax(probabilities))

    heatmap, _ = make_gradcam_heatmap(
        image_batch,
        model,
        class_index=predicted_index
    )

    resized_heatmap = tf.image.resize(
        heatmap[..., np.newaxis],
        image.shape[:2]
    ).numpy().squeeze()

    display_image = np.clip(image, 0, 1)
    colored_heatmap = plt.cm.jet(resized_heatmap)[..., :3]
    overlay = np.clip(
        0.60 * display_image + 0.40 * colored_heatmap,
        0,
        1
    )

    title = (
        f"Predicted: {class_names[predicted_index]} "
        f"({100 * probabilities[predicted_index]:.1f}%)"
    )
    if true_label is not None:
        title += f" | True: {class_names[int(true_label)]}"

    plt.figure(figsize=(12, 4))

    plt.subplot(1, 3, 1)
    plt.imshow(display_image)
    plt.title("Original image")
    plt.axis("off")

    plt.subplot(1, 3, 2)
    plt.imshow(resized_heatmap, cmap="jet")
    plt.title("Grad-CAM heatmap")
    plt.axis("off")

    plt.subplot(1, 3, 3)
    plt.imshow(overlay)
    plt.title(title)
    plt.axis("off")

    plt.tight_layout()
    plt.show()


gradcam_index = 0

if x_test is None:
    print("Normalize the images before using Grad-CAM.")
else:
    show_gradcam(
        model,
        x_test[gradcam_index],
        class_names,
        true_label=y_test[gradcam_index]
    )


## Reflection
1. Which feature best separates the expressions?
2. Did Grad-CAM focus on the mouth, eyes, or eyebrows?
3. How does randomizing colors reduce shortcut learning?
4. Was performance meaningfully above chance (25%)?
5. Describe one change you would test next.
6. Explain training, validation, and test data.
7. Why is Grad-CAM not unquestionable proof?

## Optional controlled experiment
Change exactly one item: filter count, convolution depth, augmentation, dropout, batch size, or epoch count. Compare training, validation, test, and Grad-CAM results.